# Data Preprocess 

This script concatenates most recent passive data with backup data and performs basic preprocessing on passive, EMA and Monitoring data.

In [1]:
from pathlib import Path
import sys
from datetime import date
import pandas as pd
import gc
import os
import glob
import numpy as np
import pickle

# --- Paths / imports -------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
PREPROCESSING_DIR = PROJECT_ROOT / "functions" / "preprocessing"
for p in (PROJECT_ROOT, PREPROCESSING_DIR):
    if str(p) not in sys.path:
        sys.path.append(str(p))

from server_config import (
    datapath,redcap_path,
    proj_sheet,
    preprocessed_path_jitai,
    preprocessed_path,
    raw_path,

)
from missing_data import compute_availability_metrics
from functions.preprocessing.infer_timeoffset_old_names import (
    create_utcday_tzoffset_df,
    merge_fill_tz,
)

# --- Dates ------------------------------------------------------------
today_str = date.today().strftime("%d%m%Y")
today_str = "09032026"  # For testing purposes
today_day = pd.Timestamp.today().normalize()
# --- Path -------------------------------------------------------------
datapath = Path(raw_path) / f"export_tiki_{today_str}"


## 1. Passive Data

### 1.1 Load most recent passive data

In [2]:
# actual passive + ema_data
file_pattern = os.path.join(datapath, "epoch_part*.csv")
file_list = glob.glob(file_pattern)
file_list.sort()
df_complete = pd.concat(
    (pd.read_csv(f, encoding="latin-1", low_memory=False) for f in file_list),
    ignore_index=True,
)

In [3]:
# Extract customer identifier
df_complete["customer"] = df_complete.customer.str.split("@").str.get(0)
df_complete["customer"] = df_complete["customer"].str[:4]

for col in ["startTimestamp", "endTimestamp", "createdAt"]:
    df_complete[col] = pd.to_datetime(
        df_complete[col], utc=True, errors="coerce", unit="ms"
    )


In [4]:
df_complete = df_complete[
    [
        "customer",
        "startTimestamp",
        "endTimestamp",
        "timezoneOffset",
        "type",
        "stringValue",
        "booleanValue",
        "doubleValue",
        "longValue",
        "createdAt",
    ]
]

### 1.2 Load big backup dataset

In [5]:
# Merge with backup data
backup_parquet_path = preprocessed_path + "/backup_passive_05052025.parquet"
df_backup = df_backup = pd.read_parquet(backup_parquet_path)

In [6]:
# convert booleanValue to boolean[pyarrow] dtype before concatenation so that it can be saved to .feather later
# alternative to "boolean[pyarrow]" is "boolean", but it is experimental and may change in future pandas versions
df_backup['booleanValue'] = df_backup['booleanValue'].astype('boolean[pyarrow]')
df_complete['booleanValue'] = df_complete['booleanValue'].astype('boolean[pyarrow]')

In [7]:
latest_timestamp = df_backup["startTimestamp"].max()
df_complete_filtered = df_complete[df_complete["startTimestamp"] > latest_timestamp]

### 1.3 Concat Backup and most recent data

In [8]:
df_backup_recent = pd.concat([df_backup, df_complete_filtered], ignore_index=True)

df_backup_recent["customer"] = df_backup_recent["customer"].astype("category")
df_backup_recent["type"] = df_backup_recent["type"].astype("category")

In [9]:
df_backup_recent["float_value"] = df_backup_recent["doubleValue"].fillna(
    df_backup_recent["longValue"]
)


In [10]:
# df_backup_recent = df_backup_recent.drop(columns=['doubleValue', 'longValue', 'stringValue'])


### 1.4 Create additional columns

In [11]:
# Duration in seconds
df_backup_recent["start_end"] = (
    df_backup_recent["endTimestamp"] - df_backup_recent["startTimestamp"]
).dt.total_seconds()

# Extract date and hour
df_backup_recent["startTimestamp_day"] = df_backup_recent[
    "startTimestamp"
].dt.normalize()
df_backup_recent["startTimestamp_hour"] = df_backup_recent["startTimestamp"].dt.hour


In [12]:
df_backup_recent.head()

,customer,startTimestamp,endTimestamp,timezoneOffset,type,stringValue,booleanValue,doubleValue,longValue,createdAt,float_value,start_end,startTimestamp_day,startTimestamp_hour
0,7yK3,2023-03-14 14:21:59+00:00,NaT,120.0,Latitude,None,<NA>,-53.650000,NaN,NaT,-53.650000,NaN,2023-03-14 00:00:00+00:00,14
1,7yK3,2023-03-14 14:21:59+00:00,NaT,120.0,Longitude,None,<NA>,-58.801466,NaN,NaT,-58.801466,NaN,2023-03-14 00:00:00+00:00,14
2,7yK3,2023-03-14 14:22:38+00:00,NaT,120.0,Latitude,None,<NA>,-53.650010,NaN,NaT,-53.650010,NaN,2023-03-14 00:00:00+00:00,14
3,7yK3,2023-03-14 14:22:38+00:00,NaT,120.0,Longitude,None,<NA>,-58.801466,NaN,NaT,-58.801466,NaN,2023-03-14 00:00:00+00:00,14
4,7yK3,2023-03-14 14:23:38+00:00,NaT,120.0,Latitude,None,<NA>,-53.650000,NaN,NaT,-53.650000,NaN,2023-03-14 00:00:00+00:00,14


In [13]:
df_tz = create_utcday_tzoffset_df(df_backup_recent)

2026-03-11 08:58:46,857 [INFO] [GPS] Could not infer tz for customer NGlo at 2023-11-06 00:00:00+00:00 with potential tzs [  60. -300.    0.] - gps_multiple_conflict_with_previous_no_next
2026-03-11 08:58:48,118 [INFO] [ActivityDetailCreatedAt] Could not infer tz for customer S5sH at 2024-07-16 00:00:00+00:00 with potential tzs [420. 180.] - activitydetail_multiple_conflict_with_both
2026-03-11 08:58:55,602 [INFO] [Interpolate] Could not infer tz for customer 3oNs at 2024-07-08 00:00:00+00:00 - interpolate_no_previous_no_next
2026-03-11 08:58:55,609 [INFO] [Interpolate] Could not infer tz for customer 3oNs at 2024-07-09 00:00:00+00:00 - interpolate_no_previous_no_next
2026-03-11 08:59:17,272 [INFO] [Interpolate] Could not infer tz for customer 9nSQ at 2024-04-12 00:00:00+00:00 - interpolate_no_previous_no_next
2026-03-11 08:59:28,552 [INFO] [Interpolate] Could not infer tz for customer BpVE at 2025-10-01 00:00:00+00:00 - interpolate_no_previous_no_next
2026-03-11 08:59:49,241 [INFO] [I

In [14]:
df_tz["inferred_tzoffset_timedelta"] = pd.to_timedelta(
    df_tz["inferred_tzoffset"], unit="min"
)

In [15]:
df_backup_recent = df_backup_recent.merge(
    df_tz,
    left_on=["customer", "startTimestamp_day"],
    right_on=["customer", "day"],
    how="left",
)
df_backup_recent.drop(columns=["day"], inplace=True)  # remove day from df_tz


In [16]:
# compute it once
# inferred_tzoffset = pd.to_timedelta(df_backup_recent["inferred_tzoffset"], unit="min")

df_backup_recent["local_start_time"] = (
    df_backup_recent["startTimestamp"] + df_backup_recent["inferred_tzoffset_timedelta"]
).dt.tz_localize(None)

df_backup_recent["local_end_time"] = (
    df_backup_recent["endTimestamp"] + df_backup_recent["inferred_tzoffset_timedelta"]
).dt.tz_localize(None)

df_backup_recent.head()

,customer,startTimestamp,endTimestamp,timezoneOffset,type,stringValue,booleanValue,doubleValue,longValue,createdAt,float_value,start_end,startTimestamp_day,startTimestamp_hour,inferred_tzoffset,inferred_tzoffset_source,inferred_tzoffset_timedelta,local_start_time,local_end_time
0,7yK3,2023-03-14 14:21:59+00:00,NaT,120.0,Latitude,None,<NA>,-53.650000,NaN,NaT,-53.650000,NaN,2023-03-14 00:00:00+00:00,14,120.0,gps_single,0 days 02:00:00,2023-03-14 16:21:59,NaT
1,7yK3,2023-03-14 14:21:59+00:00,NaT,120.0,Longitude,None,<NA>,-58.801466,NaN,NaT,-58.801466,NaN,2023-03-14 00:00:00+00:00,14,120.0,gps_single,0 days 02:00:00,2023-03-14 16:21:59,NaT
2,7yK3,2023-03-14 14:22:38+00:00,NaT,120.0,Latitude,None,<NA>,-53.650010,NaN,NaT,-53.650010,NaN,2023-03-14 00:00:00+00:00,14,120.0,gps_single,0 days 02:00:00,2023-03-14 16:22:38,NaT
3,7yK3,2023-03-14 14:22:38+00:00,NaT,120.0,Longitude,None,<NA>,-58.801466,NaN,NaT,-58.801466,NaN,2023-03-14 00:00:00+00:00,14,120.0,gps_single,0 days 02:00:00,2023-03-14 16:22:38,NaT
4,7yK3,2023-03-14 14:23:38+00:00,NaT,120.0,Latitude,None,<NA>,-53.650000,NaN,NaT,-53.650000,NaN,2023-03-14 00:00:00+00:00,14,120.0,gps_single,0 days 02:00:00,2023-03-14 16:23:38,NaT


In [17]:
# just to make sure that we don't use them anymore later
del df_complete_filtered
del df_complete

## 2. Monitoring data

In [18]:
df_monitoring = pd.read_csv(
    f"https://docs.google.com/spreadsheets/d/{proj_sheet}/export?format=csv"
)

In [19]:
df_monitoring = df_monitoring.copy()
df_monitoring.rename(
    columns={
        "Pseudonym": "customer",
        "EMA_ID": "ema_id",
        "Status": "status",
        "Studienversion": "study_version",
        "FOR_ID": "for_id",
        "Start EMA Baseline": "ema_base_start",
        "Ende EMA Baseline": "ema_base_end",
        "Freischaltung/ Start EMA T20": "ema_t20_start",
        "Ende EMA T20": "ema_t20_end",
        "Freischaltung/ Start EMA Post": "ema_post_start",
        "Ende EMA Post": "ema_post_end",
        "T20=Post": "t20_post",
        "Status reduziert":"status_short",
        "Dropout vor KV":"dropout_1",
        "Dropout nach KV":"dropout_2"
        
    },
    inplace=True,
)

df_monitoring = df_monitoring[
    [
        "for_id",
        "ema_id",
        "customer",
        "study_version",
        "status",
        "t20_post",
        "ema_base_start",
        "ema_base_end",
        "ema_t20_start",
        "ema_t20_end",
        "ema_post_start",
        "ema_post_end",
        "dropout_1",
        "dropout_2",
        "status_short"
    ]
]

df_monitoring["customer"] = df_monitoring["customer"].str[:4]
df_monitoring["for_id"] = df_monitoring.for_id.str.strip()

df_monitoring["ema_base_start"] = pd.to_datetime(
    df_monitoring["ema_base_start"], dayfirst=True, errors="coerce", utc=True
)
df_monitoring["ema_base_end"] = pd.to_datetime(
    df_monitoring["ema_base_end"], dayfirst=True, errors="coerce", utc=True
)

df_monitoring["study_version"] = df_monitoring["study_version"].astype("category")


### 2.1 Merge relevant columns with passive data

In [20]:
df_monitoring_short = df_monitoring[
    ["customer", "for_id", "study_version", "ema_base_start"]
]

In [21]:
data_type_groups = {
    "GPS": ["Latitude"],
    # Add more groups as needed
    "Activity": ["Steps"],
    "Sleep": ["SleepBinary"],
    "Heart_Rate": ["HeartRate"],
    # Add more groups as needed
}

In [22]:
df_monitoring_short[df_monitoring_short["customer"].isna()]

,customer,for_id,study_version,ema_base_start


In [23]:
# TODO "right vs "inner"
df_backup_recent = df_backup_recent.merge(
    df_monitoring_short, on="customer", how="right"
)
# right_merge = df_backup_recent.merge(df_monitoring_short, on="customer", how="right")
# inner_merge = df_backup_recent.merge(df_monitoring_short, on="customer", how="inner")
# df_backup_recent= df_backup_recent.merge(df_monitoring_short, on="customer", how="inner")

In [24]:
# Get a list of columns to drop (all columns not in keep_cols)
keep_cols_passive = [
    "customer",
    "for_id",
    "type",
    "startTimestamp",
    "endTimestamp",
    "start_end",
    "doubleValue",
    "longValue",
    "booleanValue",
    "startTimestamp_day",
    "startTimestamp_hour",
    "ema_base_start",
    "study_version",
    "createdAt",
    "inferred_tzoffset",
    "inferred_tzoffset_source",
    "local_start_time",
    "local_end_time",
]

df_passive_final = df_backup_recent[keep_cols_passive]

In [25]:
print("dropping these columns:")
print(set(df_backup_recent.columns) - set(df_passive_final.columns))

dropping these columns:
{'stringValue', 'timezoneOffset', 'inferred_tzoffset_timedelta', 'float_value'}


## 3. EMA data

### 3.1 Load and match relevant data from separate .csv files

In [26]:
# Beispiel: datapath = Path("/pfad/zum/verzeichnis")
session = pd.read_csv(datapath / "questionnaireSession.csv", low_memory=False)
answers = pd.read_csv(datapath / "answers.csv", low_memory=False)
choice = pd.read_csv(datapath / "choice.csv", low_memory=False)
questions = pd.read_csv(datapath / "questions.csv", low_memory=False)
questionnaire = pd.read_csv(
    datapath / "questionnaires.csv", low_memory=False
)  # ohne Komma!


In [27]:
# session data
session["user"] = session["user"].str[:4]
session.rename(
    columns={
        "user": "customer",
        "completedAt": "quest_complete",
        "createdAt": "quest_create",
        "expirationTimestamp": "quest_expir",
    },
    inplace=True,
)
# Parse epoch‑ms columns as UTC and drop tz info -> naive UTC
for col in ["quest_create", "quest_complete", "quest_expir"]:
    session[col] = pd.to_datetime(session[col], unit="ms", utc=True, errors="coerce")

df_sess = session[
    ["customer", "sessionRun", "quest_create", "quest_complete", "study", "quest_expir"]
]

In [28]:
answers["user"] = answers["user"].str[:4]
answers = answers[
    [
        "user",
        "questionnaire",
        "study",
        "question",
        "element",
        "createdAt",
        "order",
        "questionnaireSession",
    ]
]

answers["createdAt"] = pd.to_datetime(
    answers["createdAt"], unit="ms", utc=True, errors="coerce"
)

answers.rename(columns={"user": "customer", "createdAt": "quest_create"}, inplace=True)

In [29]:
# item description data
choice = choice[["element", "choice_id", "text", "question"]]
choice.rename(columns={"text": "choice_text"}, inplace=True)

In [30]:
# question description data
questions = questions[["id", "title"]]
questions.rename(columns={"id": "question", "title": "quest_title"}, inplace=True)

In [31]:
questionnaire = questionnaire[["id", "name"]]
questionnaire.rename(
    columns={"id": "questionnaire", "name": "questionnaire_name"}, inplace=True
)

In [32]:
answer_merged = pd.merge(answers, choice, on=["question", "element"])
answer_merged = pd.merge(answer_merged, questions, on="question")
answer_merged = pd.merge(answer_merged, questionnaire, on="questionnaire")
answer_merged["quest_create_day"] = answer_merged.quest_create.dt.normalize()

In [33]:
answer_merged = pd.merge(answer_merged, df_monitoring, on="customer")

### 3.2 Calculate EMA coverage

In [34]:
df_sess = pd.merge(df_sess, df_monitoring, on="customer")

In [35]:
df_sess = df_sess[
    [
        "customer",
        "sessionRun",
        "quest_create",
        "quest_complete",
        "study",
        "for_id",
        "ema_id",
        "study_version",
        "status",
        "t20_post",
        "ema_base_start",
        "ema_base_end",
        "ema_t20_start",
        "ema_t20_end",
        "ema_post_start",
        "ema_post_end",
    ]
]

In [36]:
df_sess = df_sess.copy()
df_sess["quest_create_day"] = df_sess.quest_create.dt.normalize()
df_sess["quest_complete_day"] = df_sess.quest_complete.dt.normalize()

df_sess["quest_create_hour"] = df_sess.quest_create.dt.hour
df_sess["quest_complete_hour"] = df_sess.quest_complete.dt.hour

In [37]:
# count number of completed EMA beeps in first phase
df_sess1 = df_sess.loc[df_sess.study.isin([24, 25])]
df_sess1 = df_sess1.copy()

df_sess2 = df_sess.loc[df_sess.study.isin([33, 34])]
df_sess2 = df_sess2.copy()

df_sess3 = df_sess.loc[df_sess.study.isin([38, 39])]
df_sess3 = df_sess3.copy()

In [38]:
df_sess1["quest_complete_relative1"] = (
    df_sess1["quest_complete_day"] - df_sess1["ema_base_start"]
).dt.days


sess_count1 = (
    df_sess1.dropna(subset=["quest_create"])
    .groupby("customer")["quest_create"]
    .size()
    .reset_index()
)
sess_count1 = sess_count1.rename(columns={"quest_create": "nquest_EMA1"})

# count number of completed EMA beeps in second phase
sess_count2 = (
    df_sess2.dropna(subset=["quest_create"])
    .groupby("customer")["quest_create"]
    .size()
    .reset_index()
)
sess_count2 = sess_count2.rename(columns={"quest_create": "nquest_EMA2"})

# count number of completed EMA beeps in second phase
sess_count3 = (
    df_sess3.dropna(subset=["quest_create"])
    .groupby("customer")["quest_create"]
    .size()
    .reset_index()
)
sess_count3 = sess_count3.rename(columns={"quest_create": "nquest_EMA3"})

In [39]:
df_sess = df_sess.merge(sess_count1, on=["customer"], how="left")
df_sess = df_sess.merge(sess_count2, on=["customer"], how="left")
df_sess = df_sess.merge(sess_count3, on=["customer"], how="left")

### 3.3 Calculate auxiliary variables

In [40]:
df_ema_content = answer_merged.copy()

In [41]:
# 1. Date and Time Manipulations
df_ema_content["weekday"] = df_ema_content["quest_create"].dt.day_name()
df_ema_content["createdAt_day"] = df_ema_content["quest_create"].dt.floor("D")

date_cols = ["ema_base_start", "ema_t20_start", "ema_post_start"]
for col in date_cols:
    df_ema_content[col] = pd.to_datetime(
        df_ema_content[col], dayfirst=True, errors="coerce"
    )


# 1a. Season
def get_season(month):
    if month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    elif month in [9, 10, 11]:
        return "Fall"
    else:
        return "Winter"


df_ema_content["season"] = df_ema_content["quest_create"].dt.month.apply(get_season)


# 1b. Time of Day
def get_time_of_day(hour):
    if 5 <= hour < 8:
        return "Early Morning"
    elif 8 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"


df_ema_content["time_of_day"] = df_ema_content["quest_create"].dt.hour.apply(
    get_time_of_day
)

# 2. Study Mapping / String Manipulation
study_mapping = {24: 0, 25: 0, 33: 1, 34: 1, 38: 2, 39: 2}
df_ema_content["assess"] = df_ema_content["study"].map(study_mapping)
df_ema_content["quest_title"] = df_ema_content["quest_title"].str.replace(
    "_morning", "", regex=False
)

# 3. Weekend Indicator
df_ema_content["weekend"] = (
    df_ema_content["weekday"].isin(["Saturday", "Sunday"]).astype(int)
)

# 4. Questionnaire Number
df_ema_content["quest_nr"] = (
    df_ema_content["questionnaire_name"].str.extract(r"(\d+)").astype(float)
)

# 5. Count unique questionnaires per day
df_ema_content["n_quest"] = df_ema_content.groupby(
    ["study", "customer", "createdAt_day"]
)["questionnaire_name"].transform("nunique")

# 6. Unique Day Identifier
df_ema_content["quest_nr_str"] = (
    df_ema_content["quest_nr"].fillna("unknown").astype(str)
)
df_ema_content["unique_day_id"] = (
    df_ema_content["createdAt_day"].dt.strftime("%Y%m%d")
    + "_"
    + df_ema_content["quest_nr_str"]
)

# 7. (ersetzt) Relative Start/End pro Phase & Customer
df_ema_content["ema_relative_start"] = df_ema_content.groupby(["customer", "assess"])[
    "createdAt_day"
].transform("min")
df_ema_content["ema_relative_end"] = df_ema_content.groupby(["customer", "assess"])[
    "createdAt_day"
].transform("max")

# 8. Absolute & Relative Day Index
df_ema_content["absolute_day_index"] = (
    df_ema_content["createdAt_day"] - df_ema_content["ema_relative_start"]
).dt.days + 1

df_ema_content["relative_day_index"] = (
    df_ema_content.groupby(["customer", "assess"])["createdAt_day"]
    .rank(method="dense")
    .astype(int)
)

# 9. Filter absolute_day_index > 16
max_allowed_days = 16
df_ema_content = df_ema_content[
    df_ema_content["absolute_day_index"] <= max_allowed_days
].reset_index(drop=True)

# 10. Check
high_indices = df_ema_content[df_ema_content["absolute_day_index"] > max_allowed_days]
if not high_indices.empty:
    print(
        "Warning: High absolute_day_index vorhanden:", high_indices["customer"].unique()
    )
else:
    print("All entries have absolute_day_index <= 16.")

# 11. Questionnaire Counter
df_unique = df_ema_content.drop_duplicates(
    subset=["customer", "assess", "unique_day_id"]
).copy()
df_unique["questionnaire_counter"] = (
    df_unique.groupby(["customer", "assess"]).cumcount() + 1
)
df_ema_content = df_ema_content.merge(
    df_unique[["customer", "assess", "unique_day_id", "questionnaire_counter"]],
    on=["customer", "assess", "unique_day_id"],
    how="left",
)

# 12. Missing Data behandeln
df_ema_content["assess"] = df_ema_content["assess"].fillna("unknown")
df_ema_content["absolute_day_index"] = df_ema_content["absolute_day_index"].where(
    df_ema_content["ema_relative_start"].notna(), np.nan
)

# 13. Falsche Person entfernen
filter_criteria = (
    (df_ema_content["customer"] == "UfMn")
    & (df_ema_content["study"] == 25)
    & (df_ema_content["quest_create"] > "2024-02-08")
)
df_ema_content = df_ema_content[~filter_criteria]
# **Optional: View the Updated DataFrame**
# print(df_ema_content.head())


All entries have absolute_day_index <= 16.


In [42]:
# uncomment if want to run this cell multiple times
# if "inferred_tzoffset" in df_sess.columns:
#     print("Dropping existing 'inferred_tzoffset' columns from df_sess")
#     df_sess = df_sess.drop(columns=["inferred_tzoffset", "inferred_tzoffset_source"])
# if "inferred_tzoffset" in df_ema_content.columns:
#     print("Dropping existing 'inferred_tzoffset' columns from df_ema_content")
#     df_ema_content = df_ema_content.drop(columns=["inferred_tzoffset", "inferred_tzoffset_source"])

df_sess = merge_fill_tz(
    df_sess, df_tz, day_col="quest_create_day", customer_col="customer"
)
df_ema_content = merge_fill_tz(
    df_ema_content, df_tz, day_col="quest_create_day", customer_col="customer"
)

2026-03-11 09:04:07,125 [INFO] Forward-filled timezone offsets: 640
2026-03-11 09:04:07,131 [INFO] Corrected to summer time (120 min): 85
2026-03-11 09:04:07,131 [INFO] Corrected to winter time (60 min): 65
2026-03-11 09:04:07,133 [INFO] Assumed Berlin timezone offsets: 61
2026-03-11 09:04:08,636 [INFO] Forward-filled timezone offsets: 17609
2026-03-11 09:04:08,705 [INFO] Corrected to summer time (120 min): 2617
2026-03-11 09:04:08,705 [INFO] Corrected to winter time (60 min): 4354
2026-03-11 09:04:08,730 [INFO] Assumed Berlin timezone offsets: 1954


In [43]:
# df_sess["quest_create_local"] = 

In [45]:
df_passive_final.head()

,customer,for_id,type,startTimestamp,endTimestamp,start_end,doubleValue,longValue,booleanValue,startTimestamp_day,startTimestamp_hour,ema_base_start,study_version,createdAt,inferred_tzoffset,inferred_tzoffset_source,local_start_time,local_end_time
0,4MLe,FOR11905,Steps,2023-05-17 14:44:00+00:00,2023-05-17 14:45:00+00:00,60.0,6.00,NaN,<NA>,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine,2023-05-17 15:44:00,2023-05-17 15:45:00
1,4MLe,FOR11905,ActiveBurnedCalories,2023-05-17 14:44:00+00:00,2023-05-17 14:45:00+00:00,60.0,0.14,NaN,<NA>,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine,2023-05-17 15:44:00,2023-05-17 15:45:00
2,4MLe,FOR11905,CoveredDistance,2023-05-17 14:44:00+00:00,2023-05-17 14:45:00+00:00,60.0,4.62,NaN,<NA>,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine,2023-05-17 15:44:00,2023-05-17 15:45:00
3,4MLe,FOR11905,HeartRate,2023-05-17 14:58:01+00:00,2023-05-17 14:58:38+00:00,37.0,NaN,74.0,<NA>,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine,2023-05-17 15:58:01,2023-05-17 15:58:38
4,4MLe,FOR11905,HeartRate,2023-05-17 18:00:55+00:00,NaT,NaN,NaN,95.0,<NA>,2023-05-17 00:00:00+00:00,18.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine,2023-05-17 19:00:55,NaT


## Export passive, EMA and Monitoring

In [46]:
backup_raw_path = raw_path + "/backup_passive_recent.feather"
df_passive_final.to_feather(backup_raw_path)

preprocessed_path_final = preprocessed_path_jitai + "/backup_passive_recent.feather"
df_passive_final.to_feather(preprocessed_path_final)

with open(preprocessed_path_jitai + "/ema_adherence_data.pkl", "wb") as file:
    pickle.dump(df_sess, file)


with open(preprocessed_path_jitai + "/monitoring_data.pkl", "wb") as file:
    pickle.dump(df_monitoring, file)


with open(preprocessed_path_jitai + "/ema_content.pkl", "wb") as file:
    pickle.dump(df_ema_content, file)

In [47]:

with open(preprocessed_path_jitai + "/monitoring_data.pkl", "wb") as file:
    pickle.dump(df_monitoring, file)

In [48]:
# Export df_adherence as CSV
df_sess_csv_path = preprocessed_path_jitai + "/ema_adherence_data.csv"
df_sess.to_csv(df_sess_csv_path, index=False)

# Export df_monitoring as CSV
df_monitoring_csv_path = preprocessed_path_jitai + "/monitoring_data.csv"
df_monitoring.to_csv(df_monitoring_csv_path, index=False)

# Export df_ema_content as CSV
df_ema_content_csv_path = preprocessed_path_jitai + "/ema_content.csv"
df_ema_content.to_csv(df_ema_content_csv_path, index=False)

# Export df_ema_content as CSV to freezed for data check
# df_ema_content_csv_path = preprocessed_path_freezed +'/code_check' +'/ema_content_recent.csv'
# df_ema_content.to_csv(df_ema_content_csv_path, index=False)


##### peek at exported dataframes:

## Redcap data (optional, cause not complete yet)

In [ ]:
df_redcap_final = pd.read_spss(redcap_path + "/sp1_cleaned" + "/baseline_raw_20251012.sav")

In [ ]:
df_redcap_final = df_redcap_final[[
         "for_id",
         "redcap_event_name",
         "basic_documentation_sheet_timestamp",
         "age",
         "gender",
         "scid_cv_prim_cat",
        "marital_status",
         "partnership",
         "graduation",
         "profession",
         "ema_start_date",
         "years_of_education",
         "employability",
         "ses",
         "ema_smartphone",
         "ema_sleep",
         "ema_watch",
         "prior_treatment",
         "ema_special_event",
         "psychotropic",
         "somatic_problems"]]

In [ ]:
gender_mapping = {
    "mÃ¤nnlich": "male",
    "weiblich": "female",
    "divers": "diverse",
    "kein Geschlecht": "no gender",
    "keine Angabe": "not specified",
}

scid_cv_cat_mapping = {
    "Depressive StÃ¶rung": "Depressive Disorder",
    "Spezifische Phobie": "Specific Phobia",
    "Soziale AngststÃ¶rung": "Social Anxiety Disorder",
    "Agoraphobie und/oder PanikstÃ¶rung": "Agoraphobia and/or Panic Disorder",
    "Generalisierte AngststÃ¶rung": "Generalized Anxiety Disorder",
    "ZwangsstÃ¶rung": "Obsessive-Compulsive Disorder",
    "Posttraumatische BelastungsstÃ¶rung": "Post-Traumatic Stress Disorder",
}

marital_status_mapping = {
    "ledig": "single",
    "verheiratet/eingetragene Lebenspartnerschaft": "married/registered partnership",
    "geschieden": "divorced",
    "getrennt lebend": "separated",
    "verwitwet": "widowed",
    "Sonstiges": "other",
}

employability_mapping = {
    "arbeitsfaehig": "employable",
    "arbeitsunfaehig (krankgeschrieben)": "unemployable (on sick leave)",
    "ErwerbsunfÃ¤higkeitsrente": "on disability pension",
    "Altersrente": "on retirement pension",
    "Sonstiges": "other",
}

employability_mapping_simple = {"arbeitsfaehig": "yes", "arbeitsunfaehig (krankgeschrieben)": "no", "ErwerbsunfÃ¤higkeitsrente": "no",\
"Altersrente": "no", "Sonstiges": "no"}



In [ ]:
graduation_mapping = {
    "noch Schueler:in": "still in school",
    "kein Schulabschluss": "no school degree",
    "Hauptschulabschluss oder gleichwertiger Abschluss": "elementary school degree or equivalent",
    "Realschulabschluss oder gleichwertiger Abschluss": "middle school degree or equivalent",
"Abitur/Fachhochschulreife": "high school diploma/university entrance qualification",
    "Sonstiges": "other",
}

profession_mapping = {
    "noch in  Ausbildung bzw. Studium": "still in training or studies",
    "kein Ausbildungsabschluss": "no training degree",
    "Lehre bzw. Berufsausbildung, inkl. Fachschule, Techniker": "vocational training, including technical school",
    "Universitaets- bzw. Hochschulabschluss": "university or college degree",
    "Sonstiges": "other",
}

prior_treatment_mapping = {
    "keine Vorbehandlung": "no prior treatment",
    "ambulante Psychotherapie (z.B. in einer psychotherapeutischen Ambulanz oder in einer Praxis)": "outpatient psychotherapy",
    "(teil-)stationÃ¤re Behandlung/Psychotherapie (z.B. Tagesklinik, Psychiatrie, Psychosomatische Klinik...)": "inpatient or partial inpatient treatment/psychotherapy",
    "beides": "both",
    "ja (genaue Angabe nicht vorhanden)": "yes",
}

prior_treatment_mapping_simple = {
    "keine Vorbehandlung": "no prior treatment",
    "ambulante Psychotherapie (z.B. in einer psychotherapeutischen Ambulanz oder in einer Praxis)": "prior psychotherapy",
    "(teil-)stationÃ¤re Behandlung/Psychotherapie (z.B. Tagesklinik, Psychiatrie, Psychosomatische Klinik...)": "prior inpatient",
    "beides": "prior inpatient",
    "ja (genaue Angabe nicht vorhanden)": "prior psychotherapy",
}

psychotropic_medication_mapping = {"Nein": "no", "Ja": "yes"}
somatic_mapping = {"Nein": "no", "Ja": "yes"}
ema_special_event_mapping = {0: "gewohnter Alltag", 1: "besondere Ereignisse"}


def categorize_age(age):
    if 18 <= age <= 24:
        return 0
    elif 25 <= age <= 34:
        return 1
    elif 35 <= age <= 44:
        return 2
    elif 45 <= age <= 54:
        return 3
    elif 55 <= age <= 64:
        return 4
    else:
        return 5


In [ ]:
# Apply mappings
df_redcap_final["gender_description"] = df_redcap_final["gender"].map(gender_mapping)
df_redcap_final["scid_cv_description"] = df_redcap_final["scid_cv_prim_cat"].map(
    scid_cv_cat_mapping
)
df_redcap_final["marital_status_description"] = df_redcap_final["marital_status"].map(
    marital_status_mapping
)
df_redcap_final["employability_description"] = df_redcap_final["employability"].map(
    employability_mapping
)
df_redcap_final["employability_description_simple"] = df_redcap_final[
    "employability"
].map(employability_mapping_simple)
df_redcap_final["prior_treatment_description_simple"] = df_redcap_final[
    "prior_treatment"
].map(prior_treatment_mapping_simple)
df_redcap_final["graduation_description"] = df_redcap_final["graduation"].map(
    graduation_mapping
)
df_redcap_final["profession_description"] = df_redcap_final["profession"].map(
    profession_mapping
)
df_redcap_final["prior_treatment_description"] = df_redcap_final[
    "prior_treatment"
].map(prior_treatment_mapping)

df_redcap_final["ema_special_event_description"] = df_redcap_final[
    "ema_special_event"
].map(ema_special_event_mapping)
df_redcap_final["age_description"] = df_redcap_final["age"].apply(categorize_age)
df_redcap_final["somatic_description"] = df_redcap_final["somatic_problems"].map(
    somatic_mapping
)
df_redcap_final["psychotropic_description"] = df_redcap_final["psychotropic"].map(
    psychotropic_medication_mapping
)


In [ ]:
# # add Customer ID
df_monitoring["for_id"] = df_monitoring.for_id.str.strip()
df_forid = df_monitoring[["for_id", "customer"]]

df_redcap_jitai = pd.merge(df_forid, df_redcap_final, on="for_id", how="left")

In [ ]:
df_redcap_jitai

In [ ]:
valid_df = df_redcap_jitai.dropna(subset=["ema_start_date"])


In [ ]:
with open(redcap_path + f"/redcap_data_jitai_28102025.pkl", "wb") as file:
     pickle.dump(valid_df, file)